In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_harvey_collier
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
 
# ── Styling ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   12,
    'axes.labelsize':   10,
})
BLUE   = '#2563EB'
RED    = '#DC2626'
GREEN  = '#16A34A'
ORANGE = '#EA580C'

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD & CLEAN
# ═══════════════════════════════════════════════════════════════════════════════
df = pd.read_csv(
    'cfn_volumencreditocontingente_2025_enero-septiembre.csv',
    encoding='latin1', sep=';'
)
 # Filter Sector
df = df[df['SECTOR'] != 'ACTIVIDADES FINANCIERAS Y DE SEGUROS'].copy()
     

# Rename columns to snake_case
df.columns = [
    'fecha', 'subsistema', 'entidad', 'tipo_credito', 'tipo_operacion',
    'estado_operacion', 'provincia', 'canton', 'sector', 'subsector',
    'actividad', 'num_operaciones', 'monto_otorgado'
]
 
# Drop irrelevant columns
drop_cols = ['canton', 'subsector', 'actividad', 'subsistema', 'entidad']
df.drop(columns=drop_cols, inplace=True)
 
# Parse date → quarter label
df['fecha'] = pd.to_datetime(df['fecha'], format='mixed', dayfirst=True)
df['trimestre'] = df['fecha'].dt.to_period('Q').astype(str)   # e.g. '2022Q1'
df.drop(columns=['fecha'], inplace=True)
 
# Consolidate duplicate SECTOR labels (trailing spaces / punctuation variants)
sector_map = {
    'AGRICULTURA GANADERÍA  SILVICULTURA Y PESCA':
        'AGRICULTURA GANADERÍA SILVICULTURA Y PESCA',
    'AGRICULTURA, GANADERÍA,  SILVICULTURA Y PESCA':
        'AGRICULTURA GANADERÍA SILVICULTURA Y PESCA',
    'COMERCIO AL POR MAYOR Y AL POR MENOR REPARACIÓN DE VEHÍCULOS AUTOMOTORES Y MOTOCICLETAS':
        'COMERCIO AL POR MAYOR Y MENOR',
    'COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS AUTOMOTORES Y MOTOCICLETAS':
        'COMERCIO AL POR MAYOR Y MENOR',
    'ACTIVIDADES PROFESIONALES CIENTÍFICAS Y TÉCNICAS':
        'ACTIVIDADES PROFESIONALES CIENTÍFICAS Y TÉCNICAS',
    'ACTIVIDADES PROFESIONALES, CIENTÍFICAS Y TÉCNICAS':
        'ACTIVIDADES PROFESIONALES CIENTÍFICAS Y TÉCNICAS',
    'SUMINISTRO DE ELECTRICIDAD GAS VAPOR Y AIRE ACONDICIONADO':
        'SUMINISTRO DE ELECTRICIDAD GAS VAPOR',
    'SUMINISTRO DE ELECTRICIDAD, GAS, VAPOR Y AIRE ACONDICIONADO':
        'SUMINISTRO DE ELECTRICIDAD GAS VAPOR',
}
df['sector'] = df['sector'].replace(sector_map)
 
# Log-transform target
df['log_monto'] = np.log(df['monto_otorgado'])
 
print("Dataset shape after cleaning:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nTarget stats:")
print(df[['monto_otorgado', 'log_monto']].describe().round(2))
 

Dataset shape after cleaning: (520, 9)

Columns: ['tipo_credito', 'tipo_operacion', 'estado_operacion', 'provincia', 'sector', 'num_operaciones', 'monto_otorgado', 'trimestre', 'log_monto']

Missing values:
 tipo_credito        0
tipo_operacion      0
estado_operacion    0
provincia           0
sector              0
num_operaciones     0
monto_otorgado      0
trimestre           0
log_monto           0
dtype: int64

Target stats:
       monto_otorgado  log_monto
count          520.00     520.00
mean        900844.27      12.28
std        2080278.36       1.72
min           2553.27       7.85
25%          53743.88      10.89
50%         213303.68      12.27
75%         777136.47      13.56
max       19872875.45      16.80


In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. ONE-HOT ENCODING
# ═══════════════════════════════════════════════════════════════════════════════
cat_cols = ['tipo_credito', 'tipo_operacion', 'estado_operacion',
            'provincia', 'sector', 'trimestre']
 
df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True)
 
# Keep only numeric columns for the model
X_cols = [c for c in df_model.columns if c not in ['monto_otorgado', 'log_monto']]
X = df_model[X_cols].astype(float)
y = df_model['log_monto']
 
X_const = sm.add_constant(X)
 
print(f"\nDesign matrix shape: {X_const.shape}")
 


Design matrix shape: (520, 60)


In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. FIT OLS MODEL
# ═══════════════════════════════════════════════════════════════════════════════
model = sm.OLS(y, X_const).fit()
print("\n" + "="*70)
print(model.summary())
print("="*70)
 
residuals  = model.resid
fitted     = model.fittedvalues
std_resid  = residuals / residuals.std()
influence  = model.get_influence()
leverage   = influence.hat_matrix_diag
cooks_d    = influence.cooks_distance[0]
 


                            OLS Regression Results                            
Dep. Variable:              log_monto   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.679
Method:                 Least Squares   F-statistic:                     19.57
Date:                Thu, 14 May 2026   Prob (F-statistic):           2.02e-92
Time:                        09:39:49   Log-Likelihood:                -692.67
No. Observations:                 520   AIC:                             1505.
Df Residuals:                     460   BIC:                             1761.
Df Model:                          59                                         
Covariance Type:            nonrobust                                         
                                                                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------

In [29]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4. STATISTICAL ASSUMPTION TESTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("ASSUMPTION TESTS")
print("="*70)
 
# 4a. Normality of residuals — Shapiro-Wilk & Kolmogorov-Smirnov
sw_stat, sw_p  = stats.shapiro(residuals)
ks_stat, ks_p  = stats.kstest(residuals, 'norm',
                               args=(residuals.mean(), residuals.std()))
print(f"\n[Normality] Shapiro-Wilk :  W={sw_stat:.4f},  p={sw_p:.4e}")
print(f"[Normality] Kolmogorov-Smirnov: D={ks_stat:.4f},  p={ks_p:.4e}")
 
# 4b. Homoscedasticity — Breusch-Pagan
bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(residuals, X_const)
print(f"\n[Homoscedasticity] Breusch-Pagan LM={bp_lm:.4f},  p={bp_p:.4e}")
print(f"   F-statistic={bp_f:.4f},  p={bp_fp:.4e}")
 
# 4c. Linearity — Harvey-Collier test
try:
    hc_t, hc_p = linear_harvey_collier(model)
    print(f"\n[Linearity] Harvey-Collier:  t={hc_t:.4f},  p={hc_p:.4e}")
except Exception as e:
    print(f"\n[Linearity] Harvey-Collier not applicable: {e}")
 
# 4d. Autocorrelation — Durbin-Watson
dw = sm.stats.stattools.durbin_watson(residuals)
print(f"\n[Autocorrelation] Durbin-Watson = {dw:.4f}")
print("   (values near 2 → no autocorrelation)")
 
# 4e. Multicollinearity — VIF
print("\n[Multicollinearity] VIF for numeric predictors:")
num_preds = ['num_operaciones']
vif_results = []
for col in num_preds:
    idx = list(X_const.columns).index(col)
    vif_val = variance_inflation_factor(X_const.values, idx)
    vif_results.append((col, vif_val))
    print(f"   {col}: VIF = {vif_val:.2f}")


ASSUMPTION TESTS

[Normality] Shapiro-Wilk :  W=0.9948,  p=7.3618e-02
[Normality] Kolmogorov-Smirnov: D=0.0354,  p=5.1949e-01

[Homoscedasticity] Breusch-Pagan LM=63.2285,  p=3.2946e-01
   F-statistic=1.0792,  p=3.2867e-01

[Linearity] Harvey-Collier not applicable: "The initial regressor matrix, x[:skip], issingular. You must use a value of
skip large enough to ensure that the first OLS estimator is well-defined.


[Autocorrelation] Durbin-Watson = 2.0748
   (values near 2 → no autocorrelation)

[Multicollinearity] VIF for numeric predictors:
   num_operaciones: VIF = 2.39


In [30]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5. DIAGNOSTIC PLOTS (2 figures)
# ═══════════════════════════════════════════════════════════════════════════════
 
# ── Figure 1: Four classic diagnostic plots ──────────────────────────────────
fig1, axes = plt.subplots(2, 2, figsize=(13, 10))
fig1.suptitle("OLS Diagnostics — CFN Credit Volume Model",
              fontsize=14, fontweight='bold', y=1.01)
fig1.patch.set_facecolor('#F8F9FA')
 
# (A) Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, residuals, alpha=0.55, color=BLUE, edgecolors='white',
           linewidths=0.4, s=50)
ax.axhline(0, color=RED, lw=1.5, ls='--')
lowess = sm.nonparametric.lowess(residuals, fitted, frac=0.35)
ax.plot(lowess[:, 0], lowess[:, 1], color=ORANGE, lw=2, label='LOWESS')
ax.set_xlabel('Fitted values (log monto)')
ax.set_ylabel('Residuals')
ax.set_title('(A) Residuals vs Fitted')
ax.legend(fontsize=8)
 
# (B) Normal Q-Q
ax = axes[0, 1]
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax.scatter(osm, osr, alpha=0.55, color=BLUE, edgecolors='white',
           linewidths=0.4, s=50)
ref_line = slope * np.array([osm[0], osm[-1]]) + intercept
ax.plot([osm[0], osm[-1]], ref_line, color=RED, lw=1.5, ls='--')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title(f'(B) Normal Q-Q  (R²={r**2:.3f})')
 
# (C) Scale-Location (√|std. residuals| vs Fitted)
ax = axes[1, 0]
sqrt_abs = np.sqrt(np.abs(std_resid))
ax.scatter(fitted, sqrt_abs, alpha=0.55, color=GREEN, edgecolors='white',
           linewidths=0.4, s=50)
lowess2 = sm.nonparametric.lowess(sqrt_abs, fitted, frac=0.35)
ax.plot(lowess2[:, 0], lowess2[:, 1], color=ORANGE, lw=2, label='LOWESS')
ax.set_xlabel('Fitted values (log monto)')
ax.set_ylabel('√|Standardised Residuals|')
ax.set_title('(C) Scale-Location')
ax.legend(fontsize=8)
 
# (D) Residuals vs Leverage
ax = axes[1, 1]
ax.scatter(leverage, std_resid, alpha=0.55, color=BLUE, edgecolors='white',
           linewidths=0.4, s=50)
ax.axhline(0, color='grey', lw=0.8, ls='--')
# Cook's distance contours
x_lev = np.linspace(leverage.min(), leverage.max(), 200)
p = X_const.shape[1]
for cd in [0.5, 1.0]:
    bound = np.sqrt(cd * p * (1 - x_lev) / x_lev)
    ax.plot(x_lev,  bound, color=RED, ls=':', lw=1, label=f"Cook's D={cd}")
    ax.plot(x_lev, -bound, color=RED, ls=':', lw=1)
ax.set_xlabel('Leverage')
ax.set_ylabel('Standardised Residuals')
ax.set_title("(D) Residuals vs Leverage")
ax.legend(fontsize=7)
 
plt.tight_layout()
plt.savefig('cfn_diagnostics.png', dpi=150,
            bbox_inches='tight')
print("\nFigure 1 saved → cfn_diagnostics.png")
plt.close()
 
# ── Figure 2: Distributions + Cook's D ───────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle("Residual Distribution & Influential Observations",
              fontsize=13, fontweight='bold')
fig2.patch.set_facecolor('#F8F9FA')
 
# (E) Residual histogram + normal overlay
ax = axes2[0]
ax.hist(residuals, bins=25, color=BLUE, alpha=0.75, edgecolor='white',
        density=True, label='Residuals')
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
        color=RED, lw=2, label='N(μ,σ²)')
ax.set_title('(E) Residual Distribution')
ax.set_xlabel('Residual')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
ax.text(0.05, 0.92,
        f'Shapiro-Wilk p={sw_p:.3e}\nKS p={ks_p:.3e}',
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))
 
# (F) Residuals vs num_operaciones (continuous predictor)
ax = axes2[1]
ax.scatter(df['num_operaciones'], residuals, alpha=0.55, color=GREEN,
           edgecolors='white', linewidths=0.4, s=50)
ax.axhline(0, color=RED, lw=1.5, ls='--')
lowess3 = sm.nonparametric.lowess(residuals, df['num_operaciones'], frac=0.5)
ax.plot(lowess3[:, 0], lowess3[:, 1], color=ORANGE, lw=2, label='LOWESS')
ax.set_title('(F) Residuals vs Nº Operaciones')
ax.set_xlabel('Número de Operaciones')
ax.set_ylabel('Residuals')
ax.legend(fontsize=8)
ax.text(0.05, 0.92,
        f'BP p={bp_p:.3e}',
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))
 
# (G) Cook's Distance bar chart
ax = axes2[2]
obs_idx = np.arange(len(cooks_d))
colors_c = [RED if c > 4 / len(cooks_d) else BLUE for c in cooks_d]
ax.bar(obs_idx, cooks_d, color=colors_c, width=1)
threshold = 4 / len(cooks_d)
ax.axhline(threshold, color=RED, lw=1.5, ls='--',
           label=f'Threshold 4/n={threshold:.4f}')
ax.set_title("(G) Cook's Distance")
ax.set_xlabel('Observation index')
ax.set_ylabel("Cook's D")
ax.legend(fontsize=8)
n_influential = sum(1 for c in cooks_d if c > threshold)
ax.text(0.65, 0.88,
        f'{n_influential} influential\nobservations',
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))
 
plt.tight_layout()
plt.savefig('cfn_residuals_influence.png', dpi=150,
            bbox_inches='tight')
print("Figure 2 saved → cfn_residuals_influence.png")
plt.close()
 


Figure 1 saved → cfn_diagnostics.png
Figure 2 saved → cfn_residuals_influence.png


In [31]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6. TOP SIGNIFICANT COEFFICIENTS TABLE
# ═══════════════════════════════════════════════════════════════════════════════
coef_df = pd.DataFrame({
    'coef':   model.params,
    'stderr': model.bse,
    't':      model.tvalues,
    'p':      model.pvalues,
    'ci_low': model.conf_int()[0],
    'ci_high':model.conf_int()[1],
}).drop('const').sort_values('p')
 
sig = coef_df[coef_df['p'] < 0.01]
print(f"\nSignificant predictors (p<0.01): {len(sig)} / {len(coef_df)}")
print(sig[['coef','stderr','t','p']].round(4).to_string())
 
print(f"\nModel R² = {model.rsquared:.4f}")
print(f"Adjusted R² = {model.rsquared_adj:.4f}")
print(f"F-statistic = {model.fvalue:.2f}, p = {model.f_pvalue:.4e}")
print("\nDone.")
    
    


Significant predictors (p<0.01): 12 / 59
                                                           coef  stderr        t       p
tipo_credito_PRODUCTIVO CORPORATIVO                      2.5296  0.2034  12.4360  0.0000
sector_TRANSPORTE Y ALMACENAMIENTO                      -1.3703  0.1786  -7.6716  0.0000
tipo_credito_PRODUCTIVO EMPRESARIAL                      1.2977  0.1831   7.0855  0.0000
tipo_operacion_FACTORING                                -3.0684  0.5278  -5.8141  0.0000
sector_CONSTRUCCIÓN                                      0.7891  0.2016   3.9144  0.0001
num_operaciones                                          0.0341  0.0094   3.6085  0.0003
provincia_GALAPAGOS                                      2.6231  0.7378   3.5556  0.0004
estado_operacion_ORIGINAL                               -0.9181  0.2878  -3.1903  0.0015
provincia_PASTAZA                                        3.6929  1.1690   3.1591  0.0017
sector_INDUSTRIAS MANUFACTURERAS                         0.5434  0.1